# Build and understand ResolveAI step by step

This notebook is a beginner-friendly tutorial for the customer-support agent in this project. Run cells from top to bottom. The early examples use only local sample data; cells marked **Live** call OpenAI and Pinecone and may incur API usage.

## What this application does

A customer asks a question in the Streamlit UI. LangGraph coordinates the work, Pinecone finds relevant support articles, any needed specialists review the request, and the customer receives one final answer.

```text
Customer question
  → identify topics
  → retrieve matching knowledge
  → consult relevant specialists
  → create one grounded answer
```

## Step 1 — Prepare the notebook

Run this once. It finds the project folder whether Jupyter was started inside the project or from its parent folder, then makes the `services` folder importable.

In [ ]:
# If packages are missing, run this command once in a separate cell:
# %pip install -r requirements.txt

from pathlib import Path
import sys

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / 'services').exists():
    PROJECT_DIR = PROJECT_DIR / 'customer support agent'

sys.path.insert(0, str(PROJECT_DIR))
print('Project folder:', PROJECT_DIR)

## Step 2 — Inspect the support knowledge

The app starts with a JSON knowledge base. Every article has an ID, title, category, and content. These are the records later embedded and stored in Pinecone.

In [ ]:
import json

knowledge_path = PROJECT_DIR / 'data' / 'demo_knowledge.json'
articles = json.loads(knowledge_path.read_text())

print(f'Total articles: {len(articles)}')
for article in articles[:3]:
    print(f"- {article['title']} [{article['category']}]")

## Step 3 — Try local retrieval first

The local retriever is a safe offline fallback. It looks for overlapping words between the customer question and the articles. Production uses vector similarity through Pinecone, but this lets us learn the retrieval idea without API keys.

In [ ]:
from services.retrieval import _local_search

question = 'How long does a refund take after a return?'
matches = _local_search(question, limit=3)

for match in matches:
    print(f"{match['title']} — score: {match['score']:.0%}")
    print(match['content'])
    print()

## Step 4 — Understand Pinecone ingestion

`ingest.py` reads each article, creates an OpenAI embedding, and upserts it into Pinecone. The script checks the existing index dimension first, so it produces vectors compatible with that index.

Before running the next command, fill in `OPENAI_API_KEY`, `PINECONE_API_KEY`, and `PINECONE_INDEX` in `.env`.

In [ ]:
# LIVE STEP: This creates embeddings and writes the sample articles to Pinecone.
# Run only after configuring .env.
# %run ingest.py

## Step 5 — Run a live vector search (optional)

This uses the same `search()` function as the Streamlit application. It embeds the question, queries Pinecone, and returns source metadata plus a similarity score.

In [ ]:
# LIVE STEP: requires configured OpenAI and Pinecone credentials.
from dotenv import load_dotenv
from services.retrieval import search

load_dotenv(PROJECT_DIR / '.env')
sources, retrieval_mode = search('How long does a refund take?')
print('Retrieval mode:', retrieval_mode)
for source in sources:
    print(f"- {source['title']} — {source['score']:.0%} match")

## Step 6 — Identify specialist agents

The router does not choose only one agent. It can identify all relevant domains. This matters for a question that combines an order issue and a billing issue.

In [ ]:
from services.a2a import classify_topics

multi_issue = 'My order is late and I was charged twice.'
topics = classify_topics(multi_issue)
print('Detected topics:', topics)
# Expected result: ['billing', 'order']

## Step 7 — Understand A2A handoffs

For each topic, the app consults the matching specialist. If an `A2A_*_URL` is configured, it sends the request to that external agent. Otherwise it uses a scoped built-in reviewer when an OpenAI key is available. The graph waits for every review before it writes the final answer.

In [ ]:
# LIVE STEP: invokes one specialist for each detected topic.
# Uncomment only when you want to call configured A2A or OpenAI specialists.
# from services.a2a import delegate_many
# reviews = delegate_many(topics, multi_issue, context=[])
# reviews

## Step 8 — Run the complete LangGraph workflow

This is the function the chat UI calls. It performs triage, retrieval, every required specialist review, and final-response composition. The result contains the final answer as well as useful debug information for the UI.

In [ ]:
# LIVE STEP: runs the full Pinecone + multi-agent workflow.
from services.support_graph import resolve_support_request

result = resolve_support_request(multi_issue, conversation=[])
print('Final answer:\n', result['answer'])
print('\nRetrieval mode:', result['retrieval_mode'])
print('Generation mode:', result['generation_mode'])
print('Specialists:', [review['agent'] for review in result['handoffs']])

## Step 9 — Connect this to the user interface

`app.py` sends the chat prompt to `resolve_support_request()`, then displays three things:

1. The final customer response.
2. The retrieved articles and their relevance scores.
3. The status of each specialist consultation.

Run the full application from a terminal:

```bash
streamlit run app.py
```

## Step 10 — Test ideas

Use these prompts in the app or in Step 8:

- `How long does a refund take?` — retrieval plus Billing specialist.
- `Where can I track my package?` — retrieval plus Order specialist.
- `The app crashes when I sign in.` — retrieval plus Technical specialist.
- `My order is late and I was charged twice.` — Order and Billing specialists, then one final answer.
- `I need an exception that is not in the help centre.` — low-confidence result; create a human ticket from Operations Desk.